# GOV-01 E7: focused pothole and manhole-cover improvement

E7 first creates an **automatic** error report for the completed E6 model. It then continues from E6 with a larger 320×320 image size, moderate focused sampling, and masked focal loss. This makes difficult pothole and manhole-cover examples matter more during learning.

This notebook uses training and validation data only. It never loads the protected test split.


In [ ]:
# Connect Google Drive and locate the audited data plus the completed E6 model.
from pathlib import Path
import csv
import json
import math
import shutil
import zipfile

from google.colab import drive
drive.mount('/content/drive')

DATASET_FOLDER = 'v2_multilabel_n_rdd_expansion'
CONDITIONS = ('crack', 'pothole', 'repaired_road', 'manhole_cover', 'unpaved_road', 'road_marking', 'speed_bump')
FOCUS_CONDITIONS = ('pothole', 'manhole_cover', 'repaired_road')
SEED = 42
IMAGE_SIZE = (320, 320)
BATCH_SIZE = 16
FOCUS_SAMPLE_FRACTION = 0.50
FOCAL_GAMMA = 2.0
FOCUS_POSITIVE_MULTIPLIER = 1.60

DRIVE_ROOT = Path('/content/drive/MyDrive/GOV-01')
DRIVE_DATASET_ZIP = DRIVE_ROOT / f'{DATASET_FOLDER}.zip'
DRIVE_E6_OUTPUT_ZIP = DRIVE_ROOT / 'v2_multilabel_e6_n_rdd_mobilenetv2_output.zip'
WORK_ROOT = Path('/content')
LOCAL_DATA_ROOT = WORK_ROOT / DATASET_FOLDER
LOCAL_MODEL_ROOT = WORK_ROOT / 'v2_multilabel_e7_e6_model'
E6_MODEL_NAME = 'e6_n_rdd_mobilenetv2_best.keras'
E6_MODEL_PATH = LOCAL_MODEL_ROOT / E6_MODEL_NAME
OUTPUT_DIRECTORY = WORK_ROOT / 'v2_multilabel_e7_focused_output'

if not DRIVE_DATASET_ZIP.is_file():
    raise FileNotFoundError(f'Missing dataset ZIP in Google Drive: {DRIVE_DATASET_ZIP}')
if DRIVE_E6_OUTPUT_ZIP.is_file():
    E6_OUTPUT_ZIP = DRIVE_E6_OUTPUT_ZIP
else:
    from google.colab import files
    print(f'{DRIVE_E6_OUTPUT_ZIP.name} is not in Drive. Choose the downloaded 36 MB E6 output ZIP now.')
    uploaded = files.upload()
    if DRIVE_E6_OUTPUT_ZIP.name not in uploaded:
        raise FileNotFoundError(f'Upload exactly this file: {DRIVE_E6_OUTPUT_ZIP.name}')
    E6_OUTPUT_ZIP = WORK_ROOT / DRIVE_E6_OUTPUT_ZIP.name
    E6_OUTPUT_ZIP.write_bytes(uploaded[DRIVE_E6_OUTPUT_ZIP.name])

print('Dataset ZIP:', DRIVE_DATASET_ZIP)
print('E6 model ZIP:', E6_OUTPUT_ZIP)
print('Protected test is deliberately not loaded by E7.')


In [ ]:
# Extract only what E7 needs: training and validation data, plus the best E6 model.
def split_is_complete(dataset_root, split_name):
    split_root = dataset_root / split_name
    labels_path = split_root / 'labels.csv'
    if not labels_path.is_file() or not (split_root / 'images').is_dir():
        return False
    with labels_path.open(newline='', encoding='utf-8') as source:
        rows = list(csv.DictReader(source))
    return bool(rows) and all((split_root / Path(row['materialized_image'].replace(chr(92), '/'))).is_file() for row in rows)

def training_data_is_complete(dataset_root):
    return all(split_is_complete(dataset_root, split_name) for split_name in ('train', 'validation'))

def assert_safe_members(members, destination_root):
    destination_root = destination_root.resolve()
    for member in members:
        destination = (destination_root / member.filename).resolve()
        if destination != destination_root and destination_root not in destination.parents:
            raise RuntimeError(f'Unsafe ZIP member: {member.filename}')

if training_data_is_complete(LOCAL_DATA_ROOT):
    print('Using verified training and validation data already in this Colab session.')
else:
    if LOCAL_DATA_ROOT.exists():
        print('Removing incomplete temporary data before extracting a fresh temporary copy.')
        shutil.rmtree(LOCAL_DATA_ROOT)
    print('Extracting training and validation data. The protected test folder is not extracted.')
    train_prefix = f'{DATASET_FOLDER}/train/'
    validation_prefix = f'{DATASET_FOLDER}/validation/'
    with zipfile.ZipFile(DRIVE_DATASET_ZIP) as archive:
        members = [member for member in archive.infolist() if member.filename.startswith(train_prefix) or member.filename.startswith(validation_prefix)]
        if not members:
            raise RuntimeError('The dataset ZIP has no training/validation folders at the expected location.')
        assert_safe_members(members, WORK_ROOT)
        archive.extractall(WORK_ROOT, members=members)
    if not training_data_is_complete(LOCAL_DATA_ROOT):
        raise RuntimeError('Temporary training data is incomplete. Rerun this cell once after checking the Drive ZIP.')
    print('Training and validation extraction verified:', LOCAL_DATA_ROOT)

if not E6_MODEL_PATH.is_file():
    LOCAL_MODEL_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(E6_OUTPUT_ZIP) as archive:
        member = archive.getinfo(E6_MODEL_NAME)
        assert_safe_members([member], LOCAL_MODEL_ROOT)
        archive.extract(member, LOCAL_MODEL_ROOT)
if not E6_MODEL_PATH.is_file():
    raise RuntimeError('Could not extract the selected E6 model.')

OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
print('E6 model ready:', E6_MODEL_PATH)


In [ ]:
# Read masked labels and build the image pipelines. Unknown labels remain excluded from loss and scoring.
import numpy as np
import tensorflow as tf
from sklearn.metrics import f1_score, precision_recall_fscore_support

np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow:', tf.__version__)
print('GPU devices:', gpus)
if not gpus:
    raise RuntimeError('No GPU is connected. In Colab choose Runtime -> Change runtime type -> T4 GPU, reconnect, then run from the first cell.')

def read_split(split_name):
    split_root = LOCAL_DATA_ROOT / split_name
    paths, labels, masks, sources = [], [], [], []
    with (split_root / 'labels.csv').open(newline='', encoding='utf-8') as source:
        for row in csv.DictReader(source):
            image_path = split_root / Path(row['materialized_image'].replace(chr(92), '/'))
            if not image_path.is_file():
                raise FileNotFoundError(f'Missing materialized image: {image_path}')
            label_row, mask_row = [], []
            for condition in CONDITIONS:
                known = int(row[f'{condition}_known'])
                value = row[f'{condition}_present']
                if known not in (0, 1) or (known == 0 and value != '') or (known == 1 and value not in ('0', '1')):
                    raise ValueError(f'Invalid masked label: {row["record_id"]} / {condition}')
                label_row.append(float(value) if known else 0.0)
                mask_row.append(float(known))
            paths.append(str(image_path))
            labels.append(label_row)
            masks.append(mask_row)
            sources.append(row['source_dataset'])
    return np.asarray(paths), np.asarray(labels, dtype='float32'), np.asarray(masks, dtype='float32'), np.asarray(sources)

train_paths, train_labels, train_masks, train_sources = read_split('train')
validation_paths, validation_labels, validation_masks, validation_sources = read_split('validation')
for name, labels, masks in [('Train', train_labels, train_masks), ('Validation', validation_labels, validation_masks)]:
    print(f'\n{name}: {len(labels)} images')
    for index, condition in enumerate(CONDITIONS):
        print(f'{condition:16} known={int(masks[:, index].sum()):5} positive={int((labels[:, index] * masks[:, index]).sum()):5}')

def load_image(path):
    image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMAGE_SIZE)
    return tf.cast(image, tf.float32)

def make_unbatched_dataset(paths, labels, masks, training):
    packed = np.concatenate([labels, masks], axis=1)
    dataset = tf.data.Dataset.from_tensor_slices((paths, packed))
    if training:
        dataset = dataset.shuffle(min(len(paths), 8192), seed=SEED, reshuffle_each_iteration=True)
    return dataset.map(lambda path, target: (load_image(path), target), num_parallel_calls=tf.data.AUTOTUNE)

validation_dataset = make_unbatched_dataset(validation_paths, validation_labels, validation_masks, training=False).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
validation_images = tf.data.Dataset.from_tensor_slices(validation_paths).map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


In [ ]:
# Automatic E6 mistake analysis. This is a report only; it makes no label changes.
e6_model = tf.keras.models.load_model(E6_MODEL_PATH, compile=False)
e6_probabilities = e6_model.predict(validation_images, verbose=1)

def source_error_report(condition_index):
    report = {}
    for source_name in sorted(set(validation_sources)):
        source_rows = validation_sources == source_name
        known = validation_masks[source_rows, condition_index].astype(bool)
        actual = validation_labels[source_rows, condition_index][known].astype(int)
        predicted = e6_probabilities[source_rows, condition_index][known] >= 0.50
        if len(actual) < 20 or actual.sum() < 5 or (len(actual) - actual.sum()) < 5:
            continue
        precision, recall, f1, _ = precision_recall_fscore_support(actual, predicted, average='binary', zero_division=0)
        report[source_name] = {
            'known_images': int(len(actual)), 'positive_images': int(actual.sum()),
            'precision': float(precision), 'recall': float(recall), 'f1': float(f1),
            'false_negatives': int(np.sum((actual == 1) & ~predicted)),
            'false_positives': int(np.sum((actual == 0) & predicted))
        }
    return report

error_audit = {
    'experiment': 'E6 automatic focus error audit',
    'selection_data': 'validation only',
    'protected_test_loaded': False,
    'threshold': 0.50,
    'per_source': {condition: source_error_report(CONDITIONS.index(condition)) for condition in FOCUS_CONDITIONS}
}
(OUTPUT_DIRECTORY / 'e7_e6_automatic_error_audit.json').write_text(json.dumps(error_audit, indent=2) + '\n')
for condition in FOCUS_CONDITIONS:
    print(f'\n{condition} source report (only sources with enough positive and negative validation examples):')
    for source_name, values in error_audit['per_source'][condition].items():
        print(f'  {source_name:24} F1={values["f1"]:.3f}  missed={values["false_negatives"]}  false alarms={values["false_positives"]}')


In [ ]:
# Build E7 from E6 weights at 320×320, then focus learning on difficult pothole/manhole-cover positives.
focus_indices = [CONDITIONS.index(condition) for condition in FOCUS_CONDITIONS]
focus_positive_rows = np.any((train_labels[:, focus_indices] == 1.0) & (train_masks[:, focus_indices] == 1.0), axis=1)
focus_rows = np.flatnonzero(focus_positive_rows)
other_rows = np.flatnonzero(~focus_positive_rows)
if not len(focus_rows) or not len(other_rows):
    raise RuntimeError('Focused sampling could not create both target-positive and other training groups.')
print(f'Focused-positive records: {len(focus_rows)}; other records: {len(other_rows)}')
print(f'Each E7 epoch samples about {FOCUS_SAMPLE_FRACTION:.0%} focused-positive records and {1 - FOCUS_SAMPLE_FRACTION:.0%} other records.')

focus_stream = make_unbatched_dataset(train_paths[focus_rows], train_labels[focus_rows], train_masks[focus_rows], training=True).repeat()
other_stream = make_unbatched_dataset(train_paths[other_rows], train_labels[other_rows], train_masks[other_rows], training=True).repeat()
train_dataset = tf.data.Dataset.sample_from_datasets(
    [focus_stream, other_stream], weights=[FOCUS_SAMPLE_FRACTION, 1 - FOCUS_SAMPLE_FRACTION], seed=SEED
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
steps_per_epoch = math.ceil(len(train_paths) / BATCH_SIZE)

@tf.keras.utils.register_keras_serializable(package='gov01')
def masked_targeted_focal_loss(packed_targets, probabilities):
    count = len(CONDITIONS)
    labels, masks = packed_targets[:, :count], packed_targets[:, count:]
    probabilities = tf.clip_by_value(probabilities, tf.keras.backend.epsilon(), 1.0 - tf.keras.backend.epsilon())
    base_loss = tf.keras.backend.binary_crossentropy(labels, probabilities)
    probability_of_truth = labels * probabilities + (1.0 - labels) * (1.0 - probabilities)
    focal_factor = tf.pow(1.0 - probability_of_truth, FOCAL_GAMMA)
    focus_mask = tf.constant([1.0 if condition in FOCUS_CONDITIONS else 0.0 for condition in CONDITIONS], dtype=tf.float32)
    positive_weight = 1.0 + (FOCUS_POSITIVE_MULTIPLIER - 1.0) * labels * focus_mask
    weighted_loss = base_loss * focal_factor * positive_weight * masks
    return tf.math.divide_no_nan(tf.reduce_sum(weighted_loss, axis=1), tf.reduce_sum(masks, axis=1))

source_base = e6_model.get_layer('mobilenetv2_1.00_224')
source_head = e6_model.get_layer('condition_probabilities')
base = tf.keras.applications.MobileNetV2(include_top=False, weights=None, input_shape=(*IMAGE_SIZE, 3))
base.set_weights(source_base.get_weights())
base.trainable = True
for layer in base.layers[:-40]:
    layer.trainable = False
for layer in base.layers[-40:]:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal', seed=SEED),
    tf.keras.layers.RandomRotation(0.03, seed=SEED),
    tf.keras.layers.RandomContrast(0.08, seed=SEED),
], name='e7_training_augmentation')
inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3), name='road_image')
x = augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.30, seed=SEED)(x)
outputs = tf.keras.layers.Dense(len(CONDITIONS), activation='sigmoid', name='condition_probabilities')(x)
model = tf.keras.Model(inputs, outputs, name='v2_multilabel_e7_focused_mobilenetv2_320')
model.get_layer('condition_probabilities').set_weights(source_head.get_weights())
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=8e-6), loss=masked_targeted_focal_loss)
print('E7 is initialized from E6 weights; it is not a model trained from scratch.')


In [ ]:
# Train E7 and save the checkpoint that best serves pothole, manhole cover, and repaired road.
class FocusValidationF1(tf.keras.callbacks.Callback):
    def __init__(self, images, labels, masks):
        super().__init__()
        self.images, self.labels, self.masks = images, labels, masks
        self.history = []
    def on_epoch_end(self, epoch, logs=None):
        probabilities = self.model.predict(self.images, verbose=0)
        f1_by_condition = {}
        for index, condition in enumerate(CONDITIONS):
            known = self.masks[:, index].astype(bool)
            f1_by_condition[condition] = float(f1_score(self.labels[known, index], probabilities[known, index] >= 0.50, zero_division=0))
        focus_f1 = float(np.mean([f1_by_condition[condition] for condition in FOCUS_CONDITIONS]))
        self.history.append({'focus_f1': focus_f1, **f1_by_condition})
        if logs is not None:
            logs['val_focus_f1'] = focus_f1
        print(f' validation focus F1: {focus_f1:.4f}  pothole={f1_by_condition["pothole"]:.4f}  manhole={f1_by_condition["manhole_cover"]:.4f}  repaired={f1_by_condition["repaired_road"]:.4f}')

focus_callback = FocusValidationF1(validation_images, validation_labels, validation_masks)
best_path = OUTPUT_DIRECTORY / 'e7_focused_best.keras'
history = model.fit(
    train_dataset, steps_per_epoch=steps_per_epoch, validation_data=validation_dataset, epochs=15, verbose=1,
    callbacks=[
        focus_callback,
        tf.keras.callbacks.ModelCheckpoint(best_path, monitor='val_focus_f1', mode='max', save_best_only=True),
        tf.keras.callbacks.EarlyStopping(monitor='val_focus_f1', mode='max', patience=4, restore_best_weights=True),
    ]
)
print('Completed E7 epochs:', len(history.history['loss']))


In [ ]:
# Save validation evidence and a learning chart. The protected test set remains untouched.
import matplotlib.pyplot as plt

best_model = tf.keras.models.load_model(best_path, compile=False)
probabilities = best_model.predict(validation_images, verbose=1)
per_condition = {}
for index, condition in enumerate(CONDITIONS):
    known = validation_masks[:, index].astype(bool)
    precision, recall, f1, _ = precision_recall_fscore_support(validation_labels[known, index], probabilities[known, index] >= 0.50, average='binary', zero_division=0)
    per_condition[condition] = {
        'known_images': int(known.sum()), 'positive_images': int(validation_labels[known, index].sum()),
        'precision': float(precision), 'recall': float(recall), 'f1': float(f1)
    }
metrics = {
    'experiment': 'v2_multilabel_e7_focused_pothole_manhole',
    'selection_data': 'validation only', 'protected_test_loaded': False,
    'model': 'E6-initialized MobileNetV2 at 320x320 with focused sampling and masked focal loss',
    'threshold': 0.50, 'focus_conditions': FOCUS_CONDITIONS,
    'focus_macro_f1': float(np.mean([per_condition[condition]['f1'] for condition in FOCUS_CONDITIONS])),
    'masked_validation_macro_f1': float(np.mean([item['f1'] for item in per_condition.values()])),
    'epochs_completed': len(history.history['loss']), 'per_condition': per_condition
}
(OUTPUT_DIRECTORY / 'e7_validation_metrics.json').write_text(json.dumps(metrics, indent=2) + '\n')
best_model.save(OUTPUT_DIRECTORY / 'e7_focused_final.keras')
print(json.dumps(metrics, indent=2))

figure, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['loss'], label='train loss')
axes[0].plot(history.history['val_loss'], label='validation loss')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('masked focal loss'); axes[0].legend()
axes[1].plot([item['focus_f1'] for item in focus_callback.history], label='pothole/manhole/repaired F1')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('focus F1'); axes[1].legend()
figure.tight_layout()
figure.savefig(OUTPUT_DIRECTORY / 'e7_learning_curve.png', dpi=150)
print('Protected test remains untouched.')


In [ ]:
# Keep a Drive backup and automatically download the complete E7 output ZIP.
from google.colab import files

result_zip = WORK_ROOT / 'v2_multilabel_e7_focused_pothole_manhole_output.zip'
if result_zip.exists():
    result_zip.unlink()
shutil.make_archive(str(result_zip.with_suffix('')), 'zip', OUTPUT_DIRECTORY)
drive_output = DRIVE_ROOT / result_zip.name
shutil.copy2(result_zip, drive_output)
print('Saved backup in Google Drive:', drive_output)
print('Starting browser download:', result_zip.name)
files.download(str(result_zip))
